In [ ]:
import warnings
warnings.filterwarnings('ignore')
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load Dataset
housing = pd.read_csv("Housing.csv")

# Basic Information
print("First 5 Rows:")
print(housing.head())

print("\nShape:")
print(housing.shape)

print("\nDataset Info:")
print(housing.info())

print("\nSummary Statistics:")
print(housing.describe())

# Missing Values
print("\nMissing Values (%):")
print(housing.isnull().sum() * 100 / housing.shape[0])

# Outlier Analysis
fig, axs = plt.subplots(2, 3, figsize=(12, 6))
sns.boxplot(y=housing['price'], ax=axs[0, 0])
sns.boxplot(y=housing['area'], ax=axs[0, 1])
sns.boxplot(y=housing['bedrooms'], ax=axs[0, 2])
sns.boxplot(y=housing['bathrooms'], ax=axs[1, 0])
sns.boxplot(y=housing['stories'], ax=axs[1, 1])
sns.boxplot(y=housing['parking'], ax=axs[1, 2])
plt.tight_layout()
plt.show()

# Outlier Treatment - Price
Q1 = housing['price'].quantile(0.25)
Q3 = housing['price'].quantile(0.75)
IQR = Q3 - Q1
housing = housing[
    (housing['price'] >= Q1 - 1.5 * IQR) &
    (housing['price'] <= Q3 + 1.5 * IQR)
]

# Outlier Treatment - Area
Q1 = housing['area'].quantile(0.25)
Q3 = housing['area'].quantile(0.75)
IQR = Q3 - Q1
housing = housing[
    (housing['area'] >= Q1 - 1.5 * IQR) &
    (housing['area'] <= Q3 + 1.5 * IQR)
]

# Pairplot
sns.pairplot(housing)
plt.show()

# Categorical Variables
plt.figure(figsize=(20, 12))

plt.subplot(2, 3, 1)
sns.boxplot(x='mainroad', y='price', data=housing)

plt.subplot(2, 3, 2)
sns.boxplot(x='guestroom', y='price', data=housing)

plt.subplot(2, 3, 3)
sns.boxplot(x='basement', y='price', data=housing)

plt.subplot(2, 3, 4)
sns.boxplot(x='hotwaterheating', y='price', data=housing)

plt.subplot(2, 3, 5)
sns.boxplot(x='airconditioning', y='price', data=housing)

plt.subplot(2, 3, 6)
sns.boxplot(x='furnishingstatus', y='price', data=housing)

plt.show()

# Furnishing Status vs Price
plt.figure(figsize=(10, 5))
sns.boxplot(
    x='furnishingstatus',
    y='price',
    hue='airconditioning',
    data=housing
)
plt.show()

# Binary Mapping
varlist = [
    'mainroad',
    'guestroom',
    'basement',
    'hotwaterheating',
    'airconditioning',
    'prefarea'
]

def binary_map(x):
    return x.map({'yes': 1, 'no': 0})

housing[varlist] = housing[varlist].apply(binary_map)

# Dummy Variables
status = pd.get_dummies(
    housing['furnishingstatus'],
    drop_first=True
)

housing = pd.concat(
    [housing, status],
    axis=1
)

housing.drop(
    ['furnishingstatus'],
    axis=1,
    inplace=True
)

# Train Test Split
from sklearn.model_selection import train_test_split

np.random.seed(0)

df_train, df_test = train_test_split(
    housing,
    train_size=0.7,
    random_state=100
)

# Feature Scaling
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

num_vars = [
    'area',
    'bedrooms',
    'bathrooms',
    'stories',
    'parking',
    'price'
]

df_train[num_vars] = scaler.fit_transform(
    df_train[num_vars]
)

# Correlation Heatmap
plt.figure(figsize=(16, 10))

sns.heatmap(
    df_train.corr(),
    annot=True,
    cmap="YlGnBu"
)

plt.show()

y_train = df_train.pop('price')
X_train = df_train

# Feature Selection (RFE)
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

lm = LinearRegression()

rfe = RFE(
    estimator=lm,
    n_features_to_select=6
)

rfe.fit(X_train, y_train)

selected_features = X_train.columns[rfe.support_]

print("\nSelected Features:")
print(selected_features)

# Statsmodels OLS
import statsmodels.api as sm

X_train_rfe = X_train[selected_features]

X_train_rfe = sm.add_constant(X_train_rfe)

model = sm.OLS(
    y_train,
    X_train_rfe
).fit()

print(model.summary())

# VIF Calculation
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = pd.DataFrame()

vif["Feature"] = X_train_rfe.columns

vif["VIF"] = [
    variance_inflation_factor(
        X_train_rfe.values, i
    )
    for i in range(X_train_rfe.shape[1])
]

print("\nVIF Table:")
print(vif)

# Residual Analysis
y_train_pred = model.predict(X_train_rfe)

residuals = y_train - y_train_pred

plt.figure(figsize=(8, 5))

sns.histplot(
    residuals,
    bins=20,
    kde=True
)

plt.title("Residual Distribution")
plt.xlabel("Residuals")
plt.show()

plt.figure(figsize=(8, 5))

plt.scatter(
    y_train,
    residuals
)

plt.xlabel("Actual Price")
plt.ylabel("Residuals")
plt.title("Residual Plot")

plt.show()

# Scale Test Data
df_test[num_vars] = scaler.transform(
    df_test[num_vars]
)

y_test = df_test.pop('price')

X_test = df_test

X_test = sm.add_constant(X_test)

X_test_rfe = X_test[X_train_rfe.columns]

# Predictions
y_pred = model.predict(X_test_rfe)

# Model Evaluation
from sklearn.metrics import (
    r2_score,
    mean_squared_error
)

r2 = r2_score(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = np.sqrt(mse)

print("\nModel Evaluation")
print("---------------------")
print("R² Score :", round(r2, 4))
print("MSE      :", round(mse, 4))
print("RMSE     :", round(rmse, 4))

# Actual vs Predicted
plt.figure(figsize=(8, 5))

plt.scatter(
    y_test,
    y_pred
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--'
)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted House Prices")

plt.show()